# Statistical Distributions, Hypothesis Testing & Actionable Insights


### Import Libraries

In [1]:
import sys 
import os

sys.path.insert(0, os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_3/insurance-risk-analytics'))

print('Path set. Python will now look in:', os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_3/insurance-risk-analytics'))

Path set. Python will now look in: C:\Users\dagic\OneDrive\Documents\KAIM\Week_3\insurance-risk-analytics


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, f_oneway, chi2_contingency, poisson, norm

# Reproducibility
np.random.seed(42)

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

from src.data_loader import load_data
from src.hypothesis_test import chi_squared_test, t_test

print('Libraries loaded successfully.')

Libraries loaded successfully.


### load the dataset

In [3]:
df = load_data('../data/processed/cleaned_insurance_data.csv')

df.head()

C:\Users\dagic\OneDrive\Documents\KAIM\Week_3\insurance-risk-analytics\src\data_loader.py:17: DtypeWarning: Columns (0: MaritalStatus, 1: Gender, 2: CapitalOutstanding, 3: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Dataset loaded successfully.
Shape: (1000098, 52)


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


### Claim Frequency

In [4]:
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

### Claim Severity

In [5]:
severity_df = df[df['TotalClaims'] > 0]

### Marigin

In [6]:
df['Margin'] = (df['TotalPremium'] - df['TotalClaims'])

## Hypothesis - Province Risk Difference

Hypothesis

Null Hypothesis

    -   H₀: There are no risk differences across provinces.

    -   KPI: Claim Frequency

    -   Statistical Test: Chi-Squared Test of Independence

## Segmentation
    -   Group A - Western Cape

    -   Group B - Gauteng

These are strong comparison regions because:

    -   both are economically significant,

    -   both have large policy counts,

    -   but previous EDA showed different risk behavior.

In [7]:
H ='Province Risk Difference '
H0 = 'There are no risk differences across provinces'
H1 = 'There is risk differences across provinces'

province_df = df[df['Province'].isin(['Western Cape', 'Gauteng'])]

contingency_table = pd.crosstab(province_df['Province'],province_df['HasClaim'])

province_test = chi_squared_test(contingency_table, H, H0, H1)


Province Risk Difference  CHI-SQUARED TEST
H₀: There are no risk differences across provinces
H₁: There is risk differences across provinces
---
Chi² statistic:    56.0874
Degrees of freedom: 1
P-value:            0.000000
---

=== Expected Frequencies (under H₀) ===
HasClaim             0       1
Province                      
Gauteng       392684.8  1180.2
Western Cape  170284.2   511.8


H₀: There are no risk differences across provinces is rejected for claim frequency difference between provinces. Gauteng has significantly higher claim frequency than Western Cape. This suggests that Gauteng has higher accident exposure. The company might have to give a thought of increasing premiums geographically.

## Hypothesis  — Zip Code Claim Severity Difference

Hypothesis

Null Hypothesis

    -   H₀: There are no risk differences between zip codes.

    -   KPI: Claim Severity

    -   Statistical Test: T-test


In [8]:
df['PostalCode'].value_counts(ascending=False)

PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
         ...  
7560         1
3655         1
7280         1
7760         1
6655         1
Name: count, Length: 888, dtype: int64

## Segmentation
    -   Group A - postal code = 2000

    -   Group B - zip code = 122

These are strong comparison regions because:

    -   both have large policy counts

In [9]:
zip_a = 2000
zip_b = 122

In [10]:
H = 'Zip Code Claim Severity Difference'
H0=' There is no risk differences between zip codes.'
H1 ='There is risk differences between zip codes.'

zip_df = df[df['PostalCode'].isin([zip_a, zip_b])]

claims_only = zip_df[zip_df['TotalClaims'] > 0]

group_a = claims_only[claims_only['PostalCode'] == zip_a]['TotalClaims']

group_b = claims_only[claims_only['PostalCode'] == zip_b]['TotalClaims']

zip_code_severity_test = t_test(group_a, group_b, H, H0, H1)

  Zip Code Claim Severity Difference T-TEST
H₀:  There is no risk differences between zip codes.
H₁: There is risk differences between zip codes.
---
T-statistic: 0.3854
P-value:     0.700208
Alpha:       0.05
---


H₀: There are no risk differences between zip codes is accepted. As both postal codes 2000 and 122 have an equal severity risk exposure, it is concluded that their is no severity difference between postal codes.

## Hypothesis — Margin Difference Between Zip Codes

Hypothesis

Null Hypothesis

    -   H₀: There is no significant margin difference between zip codes.

    -   KPI: Margin

    -   Statistical Test: T-test


## Segmentation
    -   Group A - postal code = 2000

    -   Group B - zip code = 122

These are strong comparison regions because:

    -   both have large policy counts

In [11]:
H = 'Margin Difference Between Zip Codes'
H0 = 'There is no risk differences between zip codes.'
H1 = 'There is risk differences between zip codes.'

group_a = df[df['PostalCode'] == zip_a]['Margin']

group_b = df[df['PostalCode'] == zip_b]['Margin']

zip_code_margin_test = t_test(group_a, group_b, H, H0, H1)


  Margin Difference Between Zip Codes T-TEST
H₀: There is no risk differences between zip codes.
H₁: There is risk differences between zip codes.
---
T-statistic: 1.1639
P-value:     0.244462
Alpha:       0.05
---


H₀: There is no significant margin difference between zip codes is accepted. Policies with postal code 2000 and 122 equal profitability .

## Hypothesis — Gender Risk Difference

Hypothesis

Null Hypothesis

    -   H₀: H₀: There is no significant risk difference between Women and Men.

    -   KPI: Claim Frequency

    -   Statistical Test: Chi-squared test


## Segmentation
    -   Group A - Male

    -   Group B - Female

In [12]:
H = 'Gender Risk Difference'
H0 = 'There is no significant risk difference between Women and Men.'
H1 = 'There is significant risk difference between Women and Men.'

gender_df = df[df['Gender'].isin(['Male', 'Female'])]

contingency_table = pd.crosstab(gender_df['Gender'],gender_df['HasClaim'])

gender_test = chi_squared_test(contingency_table, H, H0, H1)

Gender Risk Difference CHI-SQUARED TEST
H₀: There is no significant risk difference between Women and Men.
H₁: There is significant risk difference between Women and Men.
---
Chi² statistic:    0.0037
Degrees of freedom: 1
P-value:            0.951464
---

=== Expected Frequencies (under H₀) ===
HasClaim        0     1
Gender                 
Female     6740.3  14.7
Male      42723.7  93.3


H0: There is no significant risk difference between Women and Men is accepted. Both gender policy holders demonstrate an equal claim frequency .

In [13]:
results = []

results.append(province_test)
results.append(zip_code_severity_test)
results.append(zip_code_margin_test)
results.append(gender_test)


summary_df = pd.DataFrame(results)

print(summary_df)

                            Hypothesis         Test  Chi2 Statistic  \
0            Province Risk Difference   Chi-Squared       56.087384   
1   Zip Code Claim Severity Difference       T-Test             NaN   
2  Margin Difference Between Zip Codes       T-Test             NaN   
3               Gender Risk Difference  Chi-Squared        0.003705   

   Degrees of Freedom       P-Value           Decision  T-statistic  
0                 1.0  6.932050e-14          Reject H₀          NaN  
1                 NaN  7.002080e-01  Fail to Reject H₀     0.385376  
2                 NaN  2.444624e-01  Fail to Reject H₀     1.163915  
3                 1.0  9.514645e-01  Fail to Reject H₀          NaN  


## Business recommendations for rejected hypothesis

H₀: There are no risk differences across provinces is rejected for claim frequency difference between provinces.

    -   Gauteng has significantly higher claim frequency than Western Cape. 
 
    -   This suggests that Gauteng has higher accident exposure. The company might have to give a thought of increasing premiums geographically.